In [ ]:
from flask import Flask, request, render_template_string
import pandas as pd
import joblib

app = Flask(__name__)
model = joblib.load("cancer_model.pkl")

FEATURES = {
    "radius_mean": 14.0, "texture_mean": 19.0, "perimeter_mean": 92.0, "area_mean": 655.0,
    "smoothness_mean": 0.096, "compactness_mean": 0.104, "concavity_mean": 0.089,
    "concave_points_mean": 0.049, "symmetry_mean": 0.181, "fractal_dimension_mean": 0.063,
}

PAGE = """
<!doctype html>
<title>Breast Cancer</title>
<style>
  body{font-family:Arial;max-width:600px;margin:30px auto;background:#f4f6f9}
  form{background:#fff;padding:20px;border-radius:8px}
  label{display:block;margin-top:10px;font-weight:bold}
  input{width:100%;padding:7px;box-sizing:border-box}
  button{margin-top:15px;padding:10px 20px;background:#2b6cb0;color:#fff;border:0;border-radius:6px}
  .res{background:#e6fffa;padding:15px;border-radius:8px;margin-top:15px}
</style>
<h2>Breast Cancer - Logistic Regression</h2>
<form method="post">
  {% for name, default in features.items() %}
    <label>{{ name }}</label>
    <input type="number" step="any" name="{{ name }}" value="{{ vals.get(name, default) }}" required>
  {% endfor %}
  <button>Predict</button>
</form>
{% if result %}
<div class="res"><h3>Prediction: {{ result }}</h3>
  Malignant: {{ mal }}% | Benign: {{ ben }}%</div>
{% endif %}
"""

@app.route("/", methods=["GET", "POST"])
def home():
    vals, result, mal, ben = {}, None, 0, 0
    if request.method == "POST":
        vals = request.form.to_dict()
        row = pd.DataFrame([{k: float(vals[k]) for k in FEATURES}])
        p = model.predict_proba(row)[0]
        mal, ben = round(p[0] * 100, 2), round(p[1] * 100, 2)
        result = "Benign" if p[1] > p[0] else "Malignant"
    return render_template_string(PAGE, features=FEATURES, vals=vals, result=result, mal=mal, ben=ben)

if __name__ == "__main__":
    app.run(debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [13/Oct/2026 12:29:15] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Oct/2026 12:29:16] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Oct/2026 12:29:20] "POST / HTTP/1.1" 200 -
